In [1]:
import os
from pathlib import Path
import fitz
import pandas as pd
from tqdm.notebook import tqdm

In [3]:
# Change this to your project folder

PROJECT_DIR = Path.cwd()

DATA_DIR = PROJECT_DIR / "data"

HEALTHY_DIR = DATA_DIR / "Healthy"

SCHIZO_DIR = DATA_DIR / "Schizophrenia"

OUTPUT_DIR = PROJECT_DIR / "dataset"

OUTPUT_DIR.mkdir(exist_ok=True)

OUTPUT_CSV = OUTPUT_DIR / "oct_reports.csv"

In [4]:
print(PROJECT_DIR)
print()

print(HEALTHY_DIR.exists())
print(SCHIZO_DIR.exists())

C:\Users\DELL\OCT_RAG\rag_env

True
True


In [5]:
def read_pdf(pdf_path):

    text = ""

    doc = fitz.open(pdf_path)

    for page in doc:
        text += page.get_text()

    doc.close()

    return text

In [6]:
IGNORE_WORDS = [

    "Print Date",

    "Heidelberg",

    "Software Version",

    "Operator",

    "Device",

    "Comments"

]

In [7]:
def clean_text(text):

    lines = text.split("\n")

    cleaned = []

    for line in lines:

        line = line.strip()

        if len(line) == 0:
            continue

        ignore = False

        for word in IGNORE_WORDS:

            if word.lower() in line.lower():

                ignore = True
                break

        if not ignore:
            cleaned.append(line)

    return "\n".join(cleaned)

In [8]:
def process_folder(folder, label):

    records = []

    pdf_files = sorted(folder.glob("*.pdf"))

    print(f"{label} : {len(pdf_files)} PDFs")

    for pdf in tqdm(pdf_files):

        report = read_pdf(pdf)

        report = clean_text(report)

        records.append({

            "filename": pdf.name,

            "label": label,

            "report": report

        })

    return records

In [9]:
healthy = process_folder(

    HEALTHY_DIR,

    "Healthy"

)

schizo = process_folder(

    SCHIZO_DIR,

    "Schizophrenia"

)

Healthy : 47 PDFs


  0%|          | 0/47 [00:00<?, ?it/s]

Schizophrenia : 45 PDFs


  0%|          | 0/45 [00:00<?, ?it/s]

In [10]:
all_reports = healthy + schizo

df = pd.DataFrame(all_reports)

df.to_csv(

    OUTPUT_CSV,

    index=False

)

print(df.head())

print()

print("Saved to")

print(OUTPUT_CSV)

    filename    label                                             report
0   C001.pdf  Healthy  Thickness Map Single Exam Report OU\nSPECTRALI...
1   C002.pdf  Healthy  RNFL Single Exam Report OU\nSPECTRALIS® Tracki...
2  C003 .pdf  Healthy  RNFL Single Exam Report OU\nSPECTRALIS® Tracki...
3   C004.pdf  Healthy  Diagnosis:\nOD\nIR 30° ART HS]\n49/ 97\nn200 H...
4   C005.pdf  Healthy  Diagnosis:\nOD\nIR 30° ART [HS]\n4443\nL200 m\...

Saved to
C:\Users\DELL\OCT_RAG\rag_env\dataset\oct_reports.csv


In [11]:
df.shape

(92, 3)

In [12]:
df.iloc[0]["report"]

'Thickness Map Single Exam Report OU\nSPECTRALIS® Tracking Laser Tomography\nPatient:\nBHAGAT, VIKASH\nPatient ID:\n157729\nDiagnosis:\nOD\nIR 30° ART (HS]\n37 1\nL200 um\nAverage Thickness [um]\nVol [mm\n8.93\n300304\n1.59 0.48\n|Notes:\n333\n200 um\n1.76\n354\n0.56\n275\n0.22\n369\n0.58\n311\n1.64\nOCT 15° (4.5 mm) ART (10) Q: 29 [HSI\n|Date: 10/4/2025\nCenter:\nCentral Min:\n332\\ 298| Central Max:\n0.52)1.58\nSignature:\nCircle Diameters:\n1,3, 6 mm ETDRS\nDOB:\nExam.:\nComment:\nILM\n800\n700\n600\n500\n400\n300\n200\n277 um\n100\n203 um\n397 um\nRetina\nThicknes Iur\nAug/9/2003\nOct/3/2025\nIR 30° ART (HS]\n37/73\n200 um\nVol [mml\n8.88\nAverage Thickness [um]\n295332\n|1.57\\0.52\n|200 um\n331\n1.75\n337\n0.53\n253\n0.20\n358\n0.56\n316\nENGIEErINGI\n1.67\nSex:\nOCT 15° (4.5 mm) ART (9) Q: 29 THST\nM\nCenter:\nCentral Min:\n299 303 Central Max:\n0.471.61\nOS\nCircle Diameters:\n1,3, 6 mm ETDRS\nILM\n800\n700\n600\n500\n400\n300\n200\n268 um\n196 um\n318 um\n100\nRetina\nThicknes